
# 🎯 Minimax: Theory, Matrix Games, and Alpha–Beta Search (Colab Notebook)

**Audience:** Upper-division undergraduates / graduate students in CS, ML, or OR.  
**What you’ll do:**  
1. Understand **minimax** in two-player zero-sum settings.  
2. Solve a **realistic decision problem** as a **matrix game** with **linear programming** (LP).  
3. Implement **minimax with alpha–beta pruning** on Tic-Tac-Toe to see the search view.  
4. Connect the two views conceptually (LP ↔ search).

> This notebook interleaves code and explanations. You can run everything end-to-end on a free Colab GPU/CPU.


## 0) Setup

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linprog

np.set_printoptions(precision=4, suppress=True)



## 1) Minimax in Zero-Sum Games — The Big Picture

Consider two players, **Row (R)** and **Column (C)**, with a **payoff matrix** \(A\).  
- \(A_{ij}\) is the payoff to **Row** if Row plays row \(i\) and Column plays column \(j\).  
- Zero-sum means Column's payoff is \(-A_{ij}\).

**Minimax / Maximin values**
- Row wants to **maximize** her guaranteed payoff \(v\) against the worst-case Column response:  
  \[ \max_{x \in \Delta_m} \min_{j} \sum_i x_i A_{ij} \]
- Column wants to **minimize** Row’s payoff (equivalently maximize her own):  
  \[ \min_{y \in \Delta_n} \max_{i} \sum_j A_{ij} y_j \]

Here \(\Delta_m\) and \(\Delta_n\) are probability simplices (mixed strategies).  
The **Minimax Theorem** (von Neumann) says the two values are equal, and an equilibrium exists.

**Two computational views:**
1. **Matrix-game LP:** Solve a small linear program to get mixed strategies and the value \(v\).  
2. **Game-tree search:** For sequential perfect-information games (e.g., Tic-Tac-Toe), perform **minimax search** (optionally **alpha–beta pruning**) to back up values from terminal states.



## 2) Realistic Example as a Matrix Game (Ad Spend vs. Competitor Placement)

**Scenario.** A startup (**Row**) must choose one of **three** ad creatives this week (A/B/C). A dominant competitor (**Column**) picks one of **three** high-traffic venues (Search / Social / Video) to saturate. The **incremental leads** per day for Row (relative to not advertising) depend on both choices (rows vs. columns). If Column targets the same attention channel as our ad’s strength, our marginal lift is reduced.

We encode this as a payoff matrix \(A\) (Row’s payoff = expected incremental leads). Row picks a probability distribution over \{A, B, C\}; Column picks a distribution over \{Search, Social, Video\}. We compute **mixed-strategy minimax** to robustify against worst-case competitor behavior.


In [ ]:

# Payoff matrix A: rows = {Ad A, Ad B, Ad C}, columns = {Search, Social, Video}
A = np.array([
    [ 6.0,  2.0,  3.5],   # Ad A vs {Search, Social, Video}
    [ 4.0,  5.5,  2.0],   # Ad B
    [ 3.0,  2.5,  6.0]    # Ad C
], dtype=float)

row_labels = ["Ad A", "Ad B", "Ad C"]
col_labels = ["Search", "Social", "Video"]

print("Payoff matrix A (Row's payoff; Column sees -A):")
print("rows =", row_labels)
print("cols =", col_labels)
print(A)



### 2.1 Solve Row's Minimax Mixed Strategy via Linear Programming

We solve:
\[
\max_{x, v} \; v \quad \text{s.t.}\quad A^\top x \ge v \mathbf{1},\; \sum_i x_i = 1,\; x \ge 0.
\]

Interpretation: Row chooses \(x\) to **maximize the guaranteed value** \(v\); every column response yields expected payoff at least \(v\). This is solved by **minimizing \(-v\)** with linear constraints using SciPy’s `linprog`.


In [ ]:

def solve_row_minimax(A):
    # Solve Row player's minimax mixed strategy for a zero-sum game with payoff A.
    # Returns (x, v) where x is Row's mixed strategy, v is game value.
    m, n = A.shape
    # Decision vars: [x_1,...,x_m, v]; objective: minimize -v
    c = np.zeros(m + 1); c[-1] = -1.0

    # Constraints: A^T x >= v*1  <=>  -A^T x + 1*v <= 0
    A_ub = np.zeros((n, m + 1)); b_ub = np.zeros(n)
    for j in range(n):
        A_ub[j, :m] = -A[:, j]
        A_ub[j, -1] = 1.0

    # Sum-to-one: sum x_i = 1
    A_eq = np.zeros((1, m + 1)); A_eq[0, :m] = 1.0
    b_eq = np.array([1.0])

    # Bounds
    bounds = [(0, 1)] * m + [(-1e9, 1e9)]

    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    if not res.success:
        raise RuntimeError("LP failed: " + res.message)
    x = res.x[:m]; v = res.x[-1]
    return x, v

x, v = solve_row_minimax(A)

print("Row mixed strategy x:", np.round(x, 4), " (sum=", np.round(x.sum(),4), ")")
print("Guaranteed value v (expected leads/day):", np.round(v, 4))

# Sanity: expected payoff vs each Column pure action
expected_vs_cols = A.T @ x
for lbl, val in zip(col_labels, expected_vs_cols):
    print(f"Expected payoff vs Column={lbl:7s}: {val:.4f}")
print("All should be >= v up to numerical tolerance.")



### 2.2 Column’s Minimax (dual) — Verification of Minimax Equality

Column solves:
\[
\min_{y, w} \; w \quad \text{s.t.}\quad A y \le w \mathbf{1},\; \sum_j y_j = 1,\; y \ge 0.
\]
We verify Row’s \(v\) ≈ Column’s \(w\).


In [ ]:

def solve_col_minimax(A):
    # Solve Column player's minimax mixed strategy for zero-sum game with payoff A to Row.
    # Returns (y, w) where y is Column's mixed strategy, w is game value (Row's payoff).
    m, n = A.shape
    # Decision vars: [y_1,...,y_n, w]; objective: minimize w
    c = np.zeros(n + 1); c[-1] = 1.0

    # Constraints: A y <= w*1  <=>  A y - 1*w <= 0
    A_ub = np.zeros((m, n + 1)); b_ub = np.zeros(m)
    for i in range(m):
        A_ub[i, :n] = A[i, :]
        A_ub[i, -1] = -1.0

    # Sum-to-one: sum y_j = 1
    A_eq = np.zeros((1, n + 1)); A_eq[0, :n] = 1.0
    b_eq = np.array([1.0])

    bounds = [(0, 1)] * n + [(-1e9, 1e9)]

    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    if not res.success:
        raise RuntimeError("LP failed: " + res.message)
    y = res.x[:n]; w = res.x[-1]
    return y, w

y, w = solve_col_minimax(A)

print("Column mixed strategy y:", np.round(y, 4), " (sum=", np.round(y.sum(),4), ")")
print("Minimax value w (Row's payoff):", np.round(w, 4))
print("v vs w:", (v, w))

# Sanity check: Row's expected payoff for each of her pure actions vs y
expected_vs_rows = A @ y
for lbl, val in zip(row_labels, expected_vs_rows):
    print(f"Row's payoff if playing {lbl:4s} vs y: {val:.4f}")
print("All should be <= w up to numerical tolerance.")



## 3) Minimax with Alpha–Beta Pruning (Tic-Tac-Toe)

We now implement **search-based** minimax with **alpha–beta pruning**, returning the game-theoretic value (+1 = X win, -1 = O win, 0 = draw) and a best action.


In [ ]:

from functools import lru_cache
import numpy as np

EMPTY, X, O = 0, 1, -1  # X is Max, O is Min

def check_winner(board):
    b = np.array(board).reshape(3,3)
    lines = []
    lines.extend(list(b))                 # rows
    lines.extend(list(b.T))               # cols
    lines.append(b.diagonal())
    lines.append(np.fliplr(b).diagonal())
    for ln in lines:
        s = ln.sum()
        if s == 3:  return X, True
        if s == -3: return O, True
    if not (b == EMPTY).any():
        return 0, True   # draw
    return 0, False

def actions(board):
    return [i for i, v in enumerate(board) if v == EMPTY]

def apply(board, a, player):
    b = list(board); b[a] = player; return tuple(b)

@lru_cache(maxsize=None)
def minimax(board, player, alpha=-np.inf, beta=np.inf):
    winner, terminal = check_winner(board)
    if terminal:
        return (float(winner), None)

    if player == X:  # Max node
        value, best_a = -np.inf, None
        for a in actions(board):
            v, _ = minimax(apply(board, a, X), O, alpha, beta)
            if v > value:
                value, best_a = v, a
            alpha = max(alpha, value)
            if beta <= alpha:
                break
        return value, best_a
    else:            # Min node
        value, best_a = np.inf, None
        for a in actions(board):
            v, _ = minimax(apply(board, a, O), X, alpha, beta)
            if v < value:
                value, best_a = v, a
            beta = min(beta, value)
            if beta <= alpha:
                break
        return value, best_a

def pretty(board):
    syms = {X:"X", O:"O", EMPTY:"."}
    b = np.array([syms[v] for v in board]).reshape(3,3)
    print("
".join(" ".join(row) for row in b))

# Example state (X to move)
start = (X, O, X,
         O, X, EMPTY,
         EMPTY, O, EMPTY)
print("Start position (X to play):")
pretty(start)
val, move = minimax(start, X)
print("
Minimax value (X perspective):", val)
print("Best move index:", move)
print("Resulting board:")
pretty(apply(start, move, X))



## 4) LP vs. Search — A Conceptual Bridge

- **Matrix game (LP):** Simultaneous move; strategic uncertainty → **randomization in mixed strategies**.  
- **Search (alpha–beta):** Sequential, perfect information; value is computed by **backing up terminal evaluations**.

Both compute a **minimax value**, reflecting **rational adversarial play**. In extensive-form games with imperfect information, solving is harder, but the minimax idea generalizes (e.g., sequence-form LPs, CFR in poker).

---

### Optional Exercises
- Perturb entries in \(A\) and resolve; observe support changes in \(x\) and \(y\).  
- Count nodes expanded with and without alpha–beta on random boards.  
- Replace terminal-only evaluation with a heuristic and restrict depth; compare strength.
